# Сырые данные: что лежит в файлах

Задача — для каждого тестового запроса выбрать 50 объявлений. Качество оценивают по Recall@50: какая доля правильных объявлений, то есть тех, которые пользователь на самом деле выбрал по запросу, попала в наши 50; эта доля усредняется по всем запросам. Подробнее — в README.

Данные лежат в трёх файлах:

- `benchmark_items.parquet` — **корпус**: 189 212 объявлений. Ответ на тестовый запрос можно собирать только из них. Дальше в тексте этот набор так и называется — корпус.
- `benchmark_queries.parquet` — **тестовые запросы**: 2 452 запроса, для каждого нужно вернуть 50 объявлений из корпуса. Какие объявления по ним выбрали на самом деле, мы не знаем.
- `train.parquet` — **train**, данные для обучения: 497 673 строки «поля поиска → объявление, которое пользователь в итоге выбрал». Номера запроса у строк нет. Поэтому запросом мы называем набор полей поиска: текст, локация, фильтры, категория и флаг доставки. Строки train с одинаковыми полями склеиваем в один запрос. Большая часть объявлений, выбранных в train, в корпус не входит (раздел 6).

Этот ноутбук — первый просмотр файлов: какие в них колонки, где пропуски и странные значения. По итогам решаем, что исправить в данных и как их хранить. Исправления делает первый шаг решения — загрузка данных (ingestion): код лежит в `src/avito/ingestion`, запуск — `scripts/ingestion/ingest.py`. Список исправлений — в конце ноутбука.

Графики и выводы о том, как устроены запросы и локации, — в `insights.ipynb`.

Подключаем библиотеки и настраиваем, как печатаются таблицы.

In [1]:
import polars as pl

from avito import paths

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_width_chars(220);

Загружаем три файла и считаем, сколько в каждом строк и колонок.

In [2]:
train = pl.read_parquet(paths.TRAIN_RAW)
bench_queries = pl.read_parquet(paths.BENCH_QUERIES_RAW)
bench_items = pl.read_parquet(paths.BENCH_ITEMS_RAW)

pl.DataFrame({
    "файл": ["train.parquet", "benchmark_queries.parquet", "benchmark_items.parquet"],
    "строк": [train.height, bench_queries.height, bench_items.height],
    "колонок": [train.width, bench_queries.width, bench_items.width],
})

файл,строк,колонок
str,i64,i64
"""train.parquet""",497673,19
"""benchmark_queries.parquet""",2452,6
"""benchmark_items.parquet""",189212,14


## 1. Колонки

Колонки train делятся на две группы: `search_*` описывают запрос, `item_*` — выбранное по нему объявление.

| Колонка | Что в ней |
|---|---|
| `search_query` | текст запроса |
| `search_location_id` | номер локации, в которой искал пользователь: город, регион или вся страна (подробнее — в `insights.ipynb`). Названий локаций в данных нет, только номера |
| `search_is_delivery_search` | искали ли с доставкой (0/1) |
| `search_infm_params_text` | фильтры поиска одной строкой, например «Вид услуги Красота, здоровье»; может быть пустой |
| `search_category` | категория, в которой искали: 114 — «Услуги», 0 — весь Авито |
| `item_id` | идентификатор объявления |
| `item_title_raw`, `item_description_raw` | заголовок и описание |
| `item_infm_params_text` | параметры объявления одной строкой: вид и тип услуги, адрес, цены, график работы |
| `item_category_id`, `item_microcat_id` | категория и подкатегория объявления |
| `item_price`, `item_rating`, `item_rating_reviews_count` | цена, рейтинг, число отзывов |
| `item_location_id`, `item_latitude`, `item_longitude` | номер локации объявления и его координаты (широта и долгота) |
| `item_is_phone_hidden`, `item_is_message_forbidden` | скрыт ли телефон, запрещены ли сообщения |

Проверяем, что в тестовых файлах те же колонки: в `benchmark_queries` — колонки запроса и `query_id`, в `benchmark_items` — колонки объявления.

In [3]:
query_columns = [column for column in train.columns if column.startswith("search_")]
item_columns = [column for column in train.columns if column.startswith("item_")]

pl.DataFrame({
    "проверка": [
        "колонки benchmark_queries = колонки запроса из train + query_id",
        "колонки benchmark_items = колонки объявления из train",
    ],
    "результат": [
        set(bench_queries.columns) == set(query_columns) | {"query_id"},
        set(bench_items.columns) == set(item_columns),
    ],
})

проверка,результат
str,bool
"""колонки benchmark_queries = колонки запроса из train + query_id""",true
"""колонки benchmark_items = колонки объявления из train""",true


## 2. Как выглядят строки

Первые строки train: текст запроса, его локация и фильтры, заголовок и локация выбранного объявления.

In [4]:
train.select("search_query", "search_location_id", "search_infm_params_text", "item_title_raw", "item_location_id").head(5)

search_query,search_location_id,search_infm_params_text,item_title_raw,item_location_id
str,i64,str,str,i64
"""скупка телевизоров""",652430,"""""","""Скупка б/у техники""",652430
"""автоподбор""",640860,"""Рейтинг пользователя 4 звезды и выше""","""Автоподбор Разовый осмотр автомобиля""",640860
"""баня на дровах""",653240,"""Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье""","""Баня на дровах ""Прованс"" на Цветочной""",653240
"""изготовление госномера на авто""",634670,"""Вид услуги Оборудование, производство""","""Изготовление дубликатов авто номеров, гос номеров""",633570
"""укладка плитки""",658430,"""Тип услуги Ремонт квартир и домов под ключ Вид услуги Ремонт и отделка""","""Ремонт и отделка квартир под ключ""",658430


В четвёртой строке объявление выбрано не в той локации, где искали (634670 и 633570). Так бывает в 17% строк train, подробнее — в `insights.ipynb`, раздел 2.

Теперь одно объявление корпуса целиком: заголовок, параметры и начало описания. Параметры — это склейка пар «ключ значение» без разделителей: вид и тип услуги, адрес, тип цены, график работы по дням.

In [5]:
row = bench_items.row(0, named=True)

print("ЗАГОЛОВОК:", row["item_title_raw"])
print()
print("ПАРАМЕТРЫ:", row["item_infm_params_text"])
print()
print("ОПИСАНИЕ:", row["item_description_raw"][:600])

ЗАГОЛОВОК: Ремонт/выкуп компьют. и ноутбуков с выездом на дом

ПАРАМЕТРЫ: Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальная цена График работы от 34200 График работы до 82800 Время работы, с 09:30 Время работы, до 23:00 Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Диагностика компьютера Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Начальная цена Тип стоимости за услугу Стоимость 1000 Услуга Замена термопасты Начальная цена Тип стоимости за услугу Стоимость 1000 Услуга Ремонт после залития Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Чистка системы охлаждения Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Установка базовых драйверов Начальная цена Тип стоимости за услугу Стоимость 1000 Услуга Восстановление удалённых файлов Начальная цена Тип стоимости за услугу Стоимость 500 Услуга Установка антивируса Начальная цена Тип с

В параметрах много служебного повтора: «Начальная цена Тип стоимости за услугу» стоит перед каждой услугой. Но есть и слова, полезные для поиска: вид услуги («Компьютерная помощь») и названия услуг («Замена термопасты», «Ремонт видеокарты»). Числа 34200 и 82800 в «График работы от … до …» — время в секундах от полуночи, то есть 9:30 и 23:00: то же время записано рядом как «Время работы, с 09:30» и «до 23:00». Как параметры готовятся для поиска, описано в `processing/text.ipynb`, раздел 3.

## 3. Пропуски

Считаем, в каких колонках трёх файлов есть пустые значения (`null`) и сколько их.

In [6]:
rows = []
for name, df in [("train", train), ("benchmark_queries", bench_queries), ("benchmark_items", bench_items)]:
    for column in df.columns:
        nulls = df[column].null_count()
        if nulls > 0:
            rows.append({"файл": name, "колонка": column, "пустых значений": nulls, "доля": nulls / df.height})

pl.DataFrame(rows)

файл,колонка,пустых значений,доля
str,str,i64,f64
"""train""","""item_rating_reviews_count""",19211,0.038602
"""train""","""item_rating""",28232,0.056728
"""train""","""item_longitude""",4,0.000008
"""train""","""item_latitude""",4,0.000008
"""benchmark_items""","""item_rating_reviews_count""",11615,0.061386
"""benchmark_items""","""item_rating""",17631,0.093181
"""benchmark_items""","""item_longitude""",1,0.000005
"""benchmark_items""","""item_latitude""",1,0.000005
"""benchmark_items""","""item_description_raw""",35,0.000185


Пропусков мало, и они понятные: у части объявлений нет рейтинга и числа отзывов, у 35 объявлений корпуса нет описания, у единиц нет координат (4 строки train и 1 объявление корпуса).

В колонках запроса (`search_*`) нет ни одного `null`. Но это не значит, что все поля запроса заполнены: запрос без фильтров записан пустой строкой, а не `null`. Это не пропуск, а осмысленное значение «пользователь не выбирал фильтров» (раздел 4).

## 4. Запрос: категория, доставка, фильтры

Одинаковые поля поиска повторяются во многих строках train. Поэтому считаем и строки train, и запросы train, то есть строки, склеенные по одинаковым полям поиска. Сначала — сколько их в каждой категории поиска:

In [7]:
train_queries = train.select(query_columns).unique()


def counts(column: str, label: str) -> pl.DataFrame:
    train_rows = train[column].value_counts().rename({"count": "строк train"})
    train_counts = train_queries[column].value_counts().rename({"count": "запросов train"})
    bench_counts = bench_queries[column].value_counts().rename({"count": "тестовых запросов"})
    result = train_rows.join(train_counts, on=column, how="full", coalesce=True)
    result = result.join(bench_counts, on=column, how="full", coalesce=True)
    return result.fill_null(0).sort(column).rename({column: label})


counts("search_category", "категория поиска")

категория поиска,строк train,запросов train,тестовых запросов
i64,u32,u32,u32
0,34,33,222
10,2,2,0
81,2,2,0
114,497635,354426,2230


И сколько поисков с доставкой:

In [8]:
counts("search_is_delivery_search", "поиск с доставкой (1 — да)")

поиск с доставкой (1 — да),строк train,запросов train,тестовых запросов
i32,u32,u32,u32
0,497661,354452,2452
1,12,11,0


Почти все запросы ищут в категории «Услуги» (114). Поиск по всему Авито (категория 0) в train почти не встречается — 34 строки, а среди тестовых запросов его 9% (222 из 2 452). Поэтому качество на таких запросах у себя почти не проверить (`validation/split.ipynb`), подробнее о них — в `insights.ipynb`. Ещё 4 строки train — из категорий 10 и 81; их так мало, что отдельно мы их не рассматриваем. Поиск с доставкой встречается в 12 строках train и ни разу в тесте, так что эта колонка для решения бесполезна.

Самые частые значения фильтров в строках train:

In [9]:
top_filters = train["search_infm_params_text"].value_counts().sort("count", descending=True).head(10)

top_filters.select(
    pl.when(pl.col("search_infm_params_text") == "").then(pl.lit("(без фильтров)")).otherwise(pl.col("search_infm_params_text")).alias("фильтры поиска"),
    pl.col("count").alias("строк train"),
)

фильтры поиска,строк train
str,u32
"""(без фильтров)""",163999
"""Вид услуги Автосервис, аренда Тип услуги Автосервис""",19654
"""Вид услуги Ремонт и отделка""",15586
"""Вид услуги Красота, здоровье""",15355
"""Вид услуги Праздники, мероприятия""",12474
"""Тип услуги автосервиса Диагностика и ремонт авто Вид услуги Автосервис, аренда Т…",11644
"""Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье""",11589
"""Вид услуги Обучение, курсы""",11080
"""Вид услуги""",10566


Доля запросов без фильтров в строках train, в запросах train и среди тестовых запросов:

In [10]:
pl.DataFrame({
    "что считаем": ["строки train", "запросы train", "тестовые запросы"],
    "всего": [train.height, train_queries.height, bench_queries.height],
    "доля без фильтров": [
        (train["search_infm_params_text"] == "").mean(),
        (train_queries["search_infm_params_text"] == "").mean(),
        (bench_queries["search_infm_params_text"] == "").mean(),
    ],
})

что считаем,всего,доля без фильтров
str,i64,f64
"""строки train""",497673,0.329532
"""запросы train""",354463,0.352203
"""тестовые запросы""",2452,0.630506


Фильтры — это чаще всего вид и тип услуги, выбранные пользователем в интерфейсе («Вид услуги Ремонт и отделка», «Тип услуги Маникюр, педикюр»). Строка «Вид услуги» без продолжения (10 566 строк train) — фильтр записан, но значение у него пустое; что за ним стояло, по данным не понять.

Без фильтров 63% тестовых запросов и только 35% запросов train (33% строк train). Значит, у большинства тестовых запросов есть только текст и локация, и решение не может рассчитывать на фильтры. Проверочные запросы — запросы из train, на которых мы замеряем качество, — тоже нужно подбирать похожими на тест; как это сделано, описано в `validation/split.ipynb`.

## 5. Странные значения в объявлениях

### Цена

Самые частые значения цены в корпусе:

In [11]:
price = bench_items["item_price"].cast(pl.Float64)

price.value_counts().sort("count", descending=True).head(8)

item_price,count
f64,u32
1000.0,20733
500.0,16372
-1.0,15248
1500.0,9504
2000.0,7671
5000.0,5882
3000.0,5881
2500.0,5206


Сколько объявлений с ценой меньше нуля, ровно 0, ровно 1 рубль и больше 10 млн:

In [12]:
pl.DataFrame({
    "цена": ["меньше 0 (все — ровно -1)", "0", "1", "больше 10 млн"],
    "объявлений": [(price < 0).sum(), (price == 0).sum(), (price == 1).sum(), (price > 10_000_000).sum()],
})

цена,объявлений
str,i64
"""меньше 0 (все — ровно -1)""",15248
"""0""",1340
"""1""",2086
"""больше 10 млн""",76


Первые восемь объявлений с ценой больше 10 млн:

In [13]:
bench_items.filter(pl.col("item_price").cast(pl.Float64) > 10_000_000).select("item_title_raw", "item_price").head(8)

item_title_raw,item_price
str,"decimal[27,15]"
"""Дом 434,4 м² на участке 10 сот.""",11200000.000000000000000
"""Выкуп Авто в Любом Состоянии""",777777777.000000000000000
"""Дом 180 м² на участке 6 сот.""",12500000.000000000000000
"""Дом из газобетона 266 м2 (коробка )""",19800000.000000000000000
"""Выкуп авто в любом состоянии после дтп""",11111111.000000000000000
"""Выкуп авто в любом состоянии Яблоновский""",777777777.000000000000000
"""Выкуп коммерческой недвижимости/квартир, быстро""",50000000.000000000000000
"""3-к. квартира, 89 м², 10/10 эт.""",15250000.000000000000000


Цена `-1` у 15 248 объявлений означает «цена не указана». Цену 0 (1 340 объявлений) и 1 рубль (2 086), вероятно, тоже ставят вместо настоящей цены, но это мы не проверяли. Среди 76 цен больше 10 млн есть дома и квартира (например, «Дом 180 м² на участке 6 сот.» за 12,5 млн), выкуп недвижимости за 50 млн и явный мусор вроде 777 777 777 и 11 111 111 у выкупа авто.

В итоговом решении цена не используется, но её пробовали как признак для реранкера. Реранкер — это модель LightGBM, которая заново упорядочивает найденные объявления и берёт первые 50; признаки — числа о запросе и объявлении, по которым она это делает (`ranking/ranker.ipynb`, раздел 5). Поэтому `-1` заменяем на пустое значение, чтобы модель не приняла его за очень низкую цену. Остальные цены, в том числе 0, 1 рубль и огромные, не трогаем.

### Рейтинг

Рейтинг `0.0` встречается у тысяч объявлений. Смотрим, сколько у них отзывов:

In [14]:
zero_rating = bench_items.filter(pl.col("item_rating") == 0)

pl.DataFrame({
    "показатель": ["объявлений с рейтингом 0.0", "из них с 0 отзывов"],
    "значение": [zero_rating.height, (zero_rating["item_rating_reviews_count"] == 0).sum()],
})

показатель,значение
str,i64
"""объявлений с рейтингом 0.0""",3317
"""из них с 0 отзывов""",3317


Все 3 317 объявлений с рейтингом `0.0` — без отзывов. Значит, `0.0` означает «рейтинга ещё нет», а не оценку ноль. Заменяем его на пустое значение: рейтинг тоже пробовали как признак реранкера, и модель не должна принимать отсутствие отзывов за самую низкую оценку.

### Координаты

Россия лежит примерно в широтах 41–82° и долготах 19–180°. Берём объявления с координатами вне этих границ и делим их на две группы: координаты ровно (0, 0) и все остальные. Для каждой группы смотрим, сколько в ней объявлений и в каком диапазоне номера их локаций.

In [15]:
coords = bench_items.select(
    "item_title_raw",
    "item_location_id",
    pl.col("item_latitude").cast(pl.Float64),
    pl.col("item_longitude").cast(pl.Float64),
)
outside = coords.filter((pl.col("item_latitude") < 41) | (pl.col("item_longitude") < 19))
zero = outside.filter((pl.col("item_latitude") == 0) & (pl.col("item_longitude") == 0))
foreign = outside.filter(pl.col("item_latitude") != 0)

pl.DataFrame({
    "группа": ["координаты ровно (0, 0)", "другие координаты за границей"],
    "объявлений": [zero.height, foreign.height],
    "наименьший номер локации": [zero["item_location_id"].min(), foreign["item_location_id"].min()],
    "наибольший номер локации": [zero["item_location_id"].max(), foreign["item_location_id"].max()],
})

группа,объявлений,наименьший номер локации,наибольший номер локации
str,i64,i64,i64
"""координаты ровно (0, 0)""",61,621560,661420
"""другие координаты за границей""",108,900186,905802


Шесть объявлений из второй группы:

In [16]:
foreign.select("item_title_raw", "item_location_id", "item_latitude", "item_longitude").head(6)

item_title_raw,item_location_id,item_latitude,item_longitude
str,i64,f64,f64
"""Туры и экскурсии по Армении""",900358,40.177628,44.512555
"""Обучение охрана труда, рабочие профессии""",901494,31.780432,35.238474
"""Польский язык""",903433,52.736271,15.219302
"""Тату мастер(художественная татуировка в Туле)""",902972,14.856251,-2.643652
"""Фотограф в Анталии""",904467,36.886197,30.708541
"""Фотограф Пхукет""",904282,7.877035,98.396866


У 61 объявления координаты ровно (0, 0), то есть координат нет. Номера их локаций (от 621 560 до 661 420) из того же диапазона, что у локаций в примерах train выше (например, 640 860 и 658 430). Похоже, это обычные локации, у которых просто потеряны координаты. Заменяем (0, 0) на пустые значения. Для решения это важно: по координатам реранкер считает расстояние от объявления до центра локации поиска (`ranking/ranker.ipynb`, раздел 2), а точка (0, 0) лежит в Атлантическом океане.

Остальные 108 объявлений за границей разные. Среди шести показанных есть настоящие зарубежные: туры по Армении, фотографы в Анталии и на Пхукете, польский язык в Польше. Но у тату-мастера «в Туле» координаты в Западной Африке, то есть локация указана неверно. Номера локаций у всей этой группы — от 900 186 до 905 802, отдельно от номеров первой группы. Таких объявлений 108 из 189 212, поэтому мы их не трогаем.

## 6. Объявление одинаково во всех строках train

Одно объявление могут выбирать по разным запросам, поэтому оно может встречаться в нескольких строках train. За время сбора данных его могли отредактировать. Если поля не меняются, объявление можно хранить один раз, а не в каждой строке. Проверяем две вещи: меняются ли поля объявления между строками train с одним `item_id` и совпадают ли они с корпусом у объявлений, которые есть и в train, и в корпусе (их 18 142, см. следующую таблицу).

In [17]:
item_attributes = [column for column in item_columns if column != "item_id"]
per_item = train.group_by("item_id").agg([pl.col(column).n_unique() for column in item_attributes])

first_rows = train.unique(subset="item_id", keep="first").select(item_columns)
both = first_rows.join(bench_items, on="item_id", suffix="_corpus")

pl.DataFrame({
    "колонка": item_attributes,
    "объявлений train, у которых значение меняется между строками": [(per_item[column] > 1).sum() for column in item_attributes],
    "объявлений, у которых значение в train и в корпусе разное": [(both[column] != both[column + "_corpus"]).sum() for column in item_attributes],
})

колонка,"объявлений train, у которых значение меняется между строками","объявлений, у которых значение в train и в корпусе разное"
str,i64,i64
"""item_title_raw""",0,0
"""item_rating_reviews_count""",0,0
"""item_rating""",0,0
"""item_price""",0,0
"""item_microcat_id""",0,0
"""item_longitude""",0,0
"""item_location_id""",0,0
"""item_latitude""",0,0
"""item_is_phone_hidden""",0,0


Сколько разных объявлений в train и в корпусе и сколько из них общих. По общим объявлениям шло сравнение с корпусом в таблице выше.

In [18]:
pl.DataFrame({
    "показатель": ["разных объявлений в train", "объявлений в корпусе", "объявлений, которые есть и в train, и в корпусе"],
    "значение": [train["item_id"].n_unique(), bench_items.height, both.height],
})

показатель,значение
str,i64
"""разных объявлений в train""",344825
"""объявлений в корпусе""",189212
"""объявлений, которые есть и в train, и в корпусе""",18142


Те же числа в долях: какая часть объявлений train есть в корпусе и какая часть корпуса в train не встречается.

In [19]:
pl.DataFrame({
    "показатель": ["доля объявлений train, которые есть в корпусе", "доля объявлений корпуса, которых нет в train"],
    "значение": [both.height / train["item_id"].n_unique(), 1 - both.height / bench_items.height],
})

показатель,значение
str,f64
"""доля объявлений train, которые есть в корпусе""",0.052612
"""доля объявлений корпуса, которых нет в train""",0.904118


Поля объявления нигде не меняются: ни между строками train, ни между train и корпусом. Значит, каждое объявление достаточно хранить один раз.

Общих объявлений у train и корпуса мало: только 5% объявлений train есть в корпусе (18 142 из 344 825), а 90% корпуса в train не выбирали ни разу. Для решения из этого следуют две вещи:

- Нельзя просто запомнить, что выбирали раньше: большинство объявлений корпуса в train не встречается, и находить их приходится по тексту и локации. Правильные ответы теста нам неизвестны, но среди них могут быть и объявления, которых в train нет.
- Проверять качество у себя можно только на тех запросах train, у которых все выбранные объявления лежат в корпусе: иначе правильный ответ невозможно найти. Таких запросов около 5% (`validation/split.ipynb`, раздел 3).

## Выводы: что исправляет загрузка данных (ingestion)

1. **Раскладывает данные по трём таблицам.** Все объявления — и корпус, и те, что встречаются только в train, — лежат в одной таблице `items`, каждое по одному разу. У каждого объявления есть флаг `in_corpus`: «да» — объявление из корпуса, среди которого ищем ответы; «нет» — оно встречалось только в train. В таблице `train` остаются поля запроса, `item_id` и номер строки `row_id`: по нему потом из train убираются строки проверочных запросов и запросов для обучения реранкера (`validation/split.ipynb`). В таблице тестовых запросов — поля запроса и `query_id`. Ни одна колонка не выбрасывается.
2. **Меняет формат чисел.** В исходных файлах цена и координаты записаны в точном десятичном формате (`Decimal`; в выводе раздела 5 он виден как `decimal[27,15]`). Считать с ним неудобно, поэтому переводим их в обычные числа с дробной частью.
3. **Заменяет заглушки на настоящие пропуски (`null`).** «Нет значения» в данных записано заглушками: цена `-1`, рейтинг `0.0` у объявлений без отзывов, координаты `(0, 0)`. Их заменяем на `null`. А отсутствующее описание (`null` у 35 объявлений корпуса), наоборот, заменяем пустым текстом, чтобы тексты объявлений можно было обрабатывать без отдельной проверки на пропуск.
4. **Не трогает** цены 0, 1 рубль и огромные цены (в итоговом решении цена не используется) и 108 объявлений с координатами за границей.